# 📊 Integrador — Relatório de Marcas Inadimplentes

Notebook pronto para rodar no Google Colab.  
Une **LivePDV** + **Efí (Pagamentos + PIX)** e gera tabela consolidada.

## ⚠️ Segurança
- Suas credenciais ficam nos **Secrets** do Colab (chave no painel lateral 🔑), nunca no notebook.
- O certificado `.p12` é enviado pelo upload temporário (vai pra `/content/`, **não fica salvo**).
- Ao terminar, feche o notebook → os arquivos somem.

## Como usar
1. Clique no menu **Runtime → Run all** OU execute célula por célula
2. Configure os **Secrets** (instruções na célula 2)
3. Faça upload do `.p12` na célula 3
4. Veja o relatório na célula final

## 1️⃣ Instalar dependências e baixar o código

In [ ]:
!pip install -q requests beautifulsoup4 lxml python-dotenv tabulate

# Baixa os módulos do GitHub diretamente
import urllib.request
BASE = 'https://raw.githubusercontent.com/leonardochor-hash/integrador/main/'
for arq in ['livepdv_client.py', 'efi_client.py', 'relatorio_inadimplentes.py']:
    urllib.request.urlretrieve(BASE + arq, arq)
    print(f'✓ baixado: {arq}')
print('Pronto!')

## 2️⃣ Configurar Secrets (credenciais)

**Clique no ícone 🔑 (Secrets) no painel esquerdo do Colab** e adicione os seguintes secrets:

| Nome | Valor |
|------|-------|
| `LIVEPDV_USERNAME` | seu usuário do LivePDV |
| `LIVEPDV_PASSWORD` | sua senha do LivePDV |
| `EFI_CLIENT_ID` | Client_Id da conta RS (da tela Detalhar) |
| `EFI_CLIENT_SECRET` | Client_Secret da conta RS |

**Importante**: marque o toggle de cada secret para **"Acesso a este notebook"** (azul/ligado).

Quando terminar, execute a célula abaixo.

In [ ]:
import os
from google.colab import userdata

# Carrega secrets de forma segura — não imprime valores!
secrets_necessarios = ['LIVEPDV_USERNAME', 'LIVEPDV_PASSWORD', 'EFI_CLIENT_ID', 'EFI_CLIENT_SECRET']
for nome in secrets_necessarios:
    try:
        valor = userdata.get(nome)
        if valor:
            os.environ[nome] = valor
            print(f'✓ {nome}: carregado ({len(valor)} caracteres)')
        else:
            print(f'✗ {nome}: VAZIO — adicione no painel 🔑')
    except Exception as e:
        print(f'✗ {nome}: ERRO — verifique se ativou "Acesso a este notebook"')

# Configurações fixas
os.environ['LIVEPDV_BASE_URL'] = 'https://expositores.moombox.com.br'
os.environ['EFI_SANDBOX'] = 'false'
os.environ['EFI_DIAS_ATRASO_MIN'] = '1'
print('\n✓ Variáveis de ambiente configuradas')

## 3️⃣ Upload do certificado PIX (.p12)

Execute a célula abaixo. Vai aparecer um botão **"Escolher arquivos"** — selecione o arquivo `.p12` que você baixou do painel Efí.

O arquivo fica em memória temporária do Colab; quando você fechar o notebook, ele desaparece.

In [ ]:
from google.colab import files
import os

print('Selecione o arquivo .p12 baixado da Efí (conta RS):')
uploaded = files.upload()

if uploaded:
    nome_arquivo = list(uploaded.keys())[0]
    caminho = f'/content/{nome_arquivo}'
    os.environ['EFI_PIX_CERT_PATH'] = caminho
    tamanho = os.path.getsize(caminho)
    print(f'\n✓ Certificado carregado: {nome_arquivo} ({tamanho} bytes)')
else:
    print('✗ Nenhum arquivo selecionado')

## 4️⃣ Converter .p12 → .pem (necessário para Python)

A biblioteca `requests` precisa do certificado em formato `.pem`. A célula abaixo converte automaticamente.

In [ ]:
import os
import subprocess

p12 = os.environ.get('EFI_PIX_CERT_PATH')
if not p12 or not os.path.exists(p12):
    print('✗ Faça o upload do .p12 primeiro (célula anterior)')
else:
    pem = p12.replace('.p12', '.pem')
    # Converte sem senha (Efí emite sem senha)
    resultado = subprocess.run(
        ['openssl', 'pkcs12', '-in', p12, '-out', pem, '-nodes', '-passin', 'pass:'],
        capture_output=True, text=True
    )
    if os.path.exists(pem) and os.path.getsize(pem) > 0:
        os.environ['EFI_PIX_CERT_PATH'] = pem
        print(f'✓ Convertido: {pem} ({os.path.getsize(pem)} bytes)')
    else:
        print('✗ Erro na conversão:')
        print(resultado.stderr)

## 5️⃣ Teste rápido: relatório com dados fictícios (mock)

Antes de bater nas APIs reais, vamos ver se o código está funcionando com dados de exemplo.

In [ ]:
!python relatorio_inadimplentes.py --mock

## 6️⃣ Relatório REAL — conta Efí RS

Agora vamos rodar com seus dados reais. Isso vai:
1. Logar no LivePDV (usando seu usuário/senha)
2. Listar boletos vencidos na Efí Pagamentos
3. Listar cobranças PIX (cobv) vencidas
4. Cruzar CNPJs e mostrar tabela consolidada

Se aparecer erro, verifique se todos os Secrets estão corretos e se o certificado foi convertido.

In [ ]:
!python relatorio_inadimplentes.py --real --csv

## 7️⃣ Baixar o CSV gerado (opcional)

Se você quiser abrir o resultado no Excel:

In [ ]:
from google.colab import files
import glob

csvs = glob.glob('relatorios/*.csv')
if csvs:
    mais_recente = max(csvs, key=lambda f: os.path.getmtime(f))
    print(f'Baixando: {mais_recente}')
    files.download(mais_recente)
else:
    print('Nenhum CSV gerado. Rode o relatório com --csv primeiro.')

## 🔒 Limpeza ao terminar

Quando acabar, feche a aba do Colab. Os arquivos temporários (.p12, .pem) são apagados automaticamente.

Os **Secrets** continuam salvos na sua conta Google — você pode revogá-los a qualquer momento no painel 🔑.

---

## 💡 Próximos passos

Para usar a segunda conta (BS), basta:
1. Trocar os valores de `EFI_CLIENT_ID` e `EFI_CLIENT_SECRET` no painel 🔑
2. Fazer upload do `.p12` da conta BS
3. Re-executar as células 5 em diante